In [ ]:
from pynput import mouse
import time
import threading

# --- НАСТРОЙКИ ---
SWITCH_DELAY = 5  # Задержка в секундах перед началом перехвата
coordinates_list = [] # Список для хранения всех зафиксированных координат

def on_click(x, y, button, pressed):
    """Функция, которая вызывается при событии клика мыши."""
    
    # Мы обрабатываем только событие нажатия (pressed=True)
    if pressed and button == mouse.Button.left:
        
        # Фиксируем координаты
        coords = (x, y)
        coordinates_list.append(coords)
        
        # Выводим в консоль
        print(f"✅ Зафиксировано: {coords}. Нажмите Ctrl+C для завершения.")
        
        # Если нужно ограничить количество кликов:
        # if len(coordinates_list) >= 5:
        #     # Если достигли нужного количества, возвращаем False для остановки
        #     return False

def start_listening(delay):
    """Инициализирует слушатель мыши после задержки."""
    
    print(f"⏳ Скрипт стартует через {delay} секунд. Переключитесь на окно MEXC.")
    time.sleep(delay)
    
    print("\n--- ОПРЕДЕЛЕНИЕ КООРДИНАТ ---")
    print("Делайте левые клики по всем нужным элементам. Координаты будут зафиксированы.")
    print("Чтобы остановить скрипт, нажмите Ctrl+C в этом окне.")

    # Создаем слушателя мыши
    with mouse.Listener(on_click=on_click) as listener:
        # Блокируем главный поток, пока слушатель активен
        listener.join()

# --- ЗАПУСК ---
if __name__ == "__main__":
    try:
        # Используем поток для слушателя
        listening_thread = threading.Thread(target=start_listening, args=(SWITCH_DELAY,))
        listening_thread.start()
        
        # Ждем завершения слушателя (Ctrl+C остановит listener.join())
        listening_thread.join() 
        
    except KeyboardInterrupt:
        # Обработка нажатия Ctrl+C
        print("\n\n--- ЗАВЕРШЕНИЕ ---")
        print("Координаты успешно собраны:")
        for i, coords in enumerate(coordinates_list):
            print(f"Точка #{i+1}: {coords}")

⏳ Скрипт стартует через 5 секунд. Переключитесь на окно MEXC.

--- ОПРЕДЕЛЕНИЕ КООРДИНАТ ---
Делайте левые клики по всем нужным элементам. Координаты будут зафиксированы.
Чтобы остановить скрипт, нажмите Ctrl+C в этом окне.
✅ Зафиксировано: (576, 658). Нажмите Ctrl+C для завершения.
✅ Зафиксировано: (858, 749). Нажмите Ctrl+C для завершения.
✅ Зафиксировано: (696, 97). Нажмите Ctrl+C для завершения.


In [ ]:
# поле ввода лота в USD (1198, 373)
# поле ввода цены стопа (1184, 531)
# кнопка Открыть Лонга (1182, 674)
# кнопка Открыть Шорт (1283, 677)

# значок открыть форму редактирования стопов (698, 659)
# редактировать первый стоп в списке (860, 451)

# поле ввода цены стопа (503, 508)
# кнопка Подтвердить (788, 655)
# конпка Отмена (858, 749)

# крестик закрытия формы редактирования стопов (821, 197)

In [5]:
import pyautogui
from PIL import Image
import pytesseract
import re
import time
import os
import logging
import cv2 
import numpy as np 

# --- ⚙️ НАСТРОЙКА ЛОГИРОВАНИЯ ---
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# --- 🛠️ ПУТЬ К TESSERACT (ОБЯЗАТЕЛЬНО) ---
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# --- ⚙️ ПАРАМЕТРЫ СКРИПТА ---

# ТЕКУЩЕЕ ОКНО РАСПОЗНАВАНИЯ
OHLC_REGION = (279, 304, 454, 19) 

CANDLE_BASE_X = 803  
CANDLE_BASE_Y = 397  
CANDLE_WIDTH = 5     

# OCR Config: Ограничиваем символы для лучшего распознавания цифр
OCR_CONFIG = '-l eng --psm 6 -c tessedit_char_whitelist=0123456789.OHLCW ' 

POLLING_INTERVAL = 5 
NUM_CANDLES = 11     
START_TIMEOUT = 5    
MAX_RUNS = 3         

# --- ФУНКЦИИ ---

def parse_ohlc_fixed_order(text: str) -> dict:
    """Парсит OHLC, полагаясь на фиксированный порядок чисел (O, H, L, C)."""
    
    # 1. Очистка текста: Удаляем все, что не является цифрами, точкой или пробелом.
    text_processed = text.replace(',', '.').upper()
    text_cleaned = re.sub(r'[^0-9\.\s]', ' ', text_processed).strip()
    
    logger.debug(f"Парсер: Очищенный текст (только числа): '{text_cleaned}'")
    
    data = {}
    
    # 2. Ищем все числа с плавающей точкой
    prices = re.findall(r'[0-9]*\.[0-9]+', text_cleaned)
    
    if len(prices) >= 4:
        logger.debug(f"Парсер: Найдено {len(prices)} чисел. Присваиваем по порядку O, H, L, C.")
        
        try:
            data['O'] = float(prices[0])
            data['H'] = float(prices[1])
            data['L'] = float(prices[2])
            data['C'] = float(prices[3])
        except ValueError:
            logger.warning("Парсер: Ошибка преобразования одного из чисел в float.")
            return {}

    return data

def recognize_candle(i: int, ohlc_region: tuple) -> dict | None:
    
    logger.info(f"\n--- Начинаем сканирование свечи {i} ---")
    
    target_x = CANDLE_BASE_X - (i * CANDLE_WIDTH) 
    target_y = CANDLE_BASE_Y 
    
    pyautogui.moveTo(target_x, target_y, duration=0.5) 
    time.sleep(0.5) 

    screenshot_pil = pyautogui.screenshot(region=ohlc_region)
    img_cv = np.array(screenshot_pil)
    img_cv = cv2.cvtColor(img_cv, cv2.COLOR_RGB2BGR) 
    
    # --- 3. ОБРАБОТКА ИЗОБРАЖЕНИЯ (Upscaling, Blur, Otsu) ---
    
    # 3.1 Увеличение масштаба (в 2 раза)
    scale_factor = 2
    img_upscaled = cv2.resize(img_cv, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_LINEAR)
    
    # 3.2 Преобразование в оттенки серого
    gray = cv2.cvtColor(img_upscaled, cv2.COLOR_BGR2GRAY)
    
    # 3.3 Небольшое размытие для сглаживания артефактов
    gray_blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    
    # 3.4 Otsu Thresholding
    _, binary = cv2.threshold(gray_blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    temp_img_path = f"temp_candle_{i}.png"
    cv2.imwrite(temp_img_path, binary)
    
    # 5. OCR: Читаем обработанное изображение
    text = pytesseract.image_to_string(temp_img_path, config=OCR_CONFIG)
    raw_text = text.strip()
    
    os.remove(temp_img_path) 
    
    logger.debug(f"Tesseract: Сырой текст после Otsu-обработки: '{raw_text}'")
    
    # Сохраняем скриншот ВСЕГДА
    screenshot_filename = f"capture_candle_{i}_{int(time.time())}.png"
    screenshot_pil.save(screenshot_filename)
    logger.debug(f"Скриншот сохранен: {screenshot_filename}")

    # 6. Парсинг и проверка
    data = parse_ohlc_fixed_order(raw_text) 
    
    if len(data) == 4:
        logger.info(f"УСПЕХ: Все 4 значения распознаны.")
        return data
    else:
        logger.warning(f"Не удалось найти все 4 значения. Найдено: {len(data)}.")
        return None

# --- ГЛАВНЫЙ ЦИКЛ ---

def run_ohlc_scanner():
    print(f"**❗ Убедитесь, что интерфейс MEXC на АНГЛИЙСКОМ языке.**")
    print(f"**❗ ОКНО РАСПОЗНАВАНИЯ: (X_min=279, Y_min=304, Width=454, Height=19).**")
    print(f"**📢 ИСПОЛЬЗУЕТСЯ УЛУЧШЕННАЯ ПРЕДОБРАБОТКА ИЗОБРАЖЕНИЯ (Upscale + Blur).**")
    print(f"⏳ Переключитесь на окно MEXC. Сканирование начнется через {START_TIMEOUT} секунд. Выполнится {MAX_RUNS} цикла.")
    time.sleep(START_TIMEOUT)
    
    for run_num in range(1, MAX_RUNS + 1):
        print(f"\n==================== ЦИКЛ СКАНИРОВАНИЯ #{run_num} ====================")
        start_time = time.time()
        
        for i in range(NUM_CANDLES):
            ohlc_data = recognize_candle(i, OHLC_REGION)
            
            if ohlc_data and all(key in ohlc_data for key in ('O', 'H', 'L', 'C')):
                print(f"✅ Свеча {i:2d} | O:{ohlc_data.get('O'):.4f} H:{ohlc_data.get('H'):.4f} L:{ohlc_data.get('L'):.4f} C:{ohlc_data.get('C'):.4f}")
            else:
                print(f"❌ Свеча {i:2d} | OHLC не найдено (или неполно). См. логи DEBUG.")
        
        elapsed_time = time.time() - start_time
        sleep_time = POLLING_INTERVAL - elapsed_time
        
        if sleep_time > 0:
            print(f"--- Ожидание {sleep_time:.2f} сек. до следующего цикла... ---")
            time.sleep(sleep_time)

    print("\n==================== СКАНИРОВАНИЕ ЗАВЕРШЕНО ====================")
    
if __name__ == "__main__":
    try:
        run_ohlc_scanner()
    except Exception as e:
        logger.critical(f"Критическая ошибка выполнения: {e}")

**❗ Убедитесь, что интерфейс MEXC на АНГЛИЙСКОМ языке.**
**❗ ОКНО РАСПОЗНАВАНИЯ: (X_min=279, Y_min=304, Width=454, Height=19).**
**📢 ИСПОЛЬЗУЕТСЯ УЛУЧШЕННАЯ ПРЕДОБРАБОТКА ИЗОБРАЖЕНИЯ (Upscale + Blur).**
⏳ Переключитесь на окно MEXC. Сканирование начнется через 5 секунд. Выполнится 1 цикла.


INFO: 
--- Начинаем сканирование свечи 0 ---



==================== ЦИКЛ СКАНИРОВАНИЯ #1 ====================


DEBUG: ['C:\\Program Files\\Tesseract-OCR\\tesseract.exe', 'C:\\Users\\abuta\\Desktop\\mexc_bot\\temp_candle_0.png', 'C:\\Users\\abuta\\AppData\\Local\\Temp\\tess_6u23gzaf', '-l', 'eng', '--psm', '6', '-c', 'tessedit_char_whitelist=0123456789.OHLCW', 'txt']
DEBUG: Tesseract: Сырой текст после Otsu-обработки: '00.5286000.5295L0.5286C0.52950.00090.175'
DEBUG: Скриншот сохранен: capture_candle_0_1762469331.png
DEBUG: Парсер: Очищенный текст (только числа): '00.5286000.5295 0.5286 0.52950.00090.175'
DEBUG: Парсер: Найдено 6 чисел. Присваиваем по порядку O, H, L, C.
INFO: УСПЕХ: Все 4 значения распознаны.
INFO: 
--- Начинаем сканирование свечи 1 ---


✅ Свеча  0 | O:0.5286 H:0.5295 L:0.5286 C:0.5295


...

✅ Свеча 58 | O:0.5285 H:0.5285 L:0.5280 C:0.5281


DEBUG: ['C:\\Program Files\\Tesseract-OCR\\tesseract.exe', 'C:\\Users\\abuta\\Desktop\\mexc_bot\\temp_candle_59.png', 'C:\\Users\\abuta\\AppData\\Local\\Temp\\tess_t1p0t4zi', '-l', 'eng', '--psm', '6', '-c', 'tessedit_char_whitelist=0123456789.OHLCW', 'txt']
DEBUG: Tesseract: Сырой текст после Otsu-обработки: '00.5284000.5286LO.5283C0.52850.00000.023'
DEBUG: Скриншот сохранен: capture_candle_59_1762469392.png
DEBUG: Парсер: Очищенный текст (только числа): '00.5284000.5286  .5283 0.52850.00000.023'
DEBUG: Парсер: Найдено 6 чисел. Присваиваем по порядку O, H, L, C.
INFO: УСПЕХ: Все 4 значения распознаны.
INFO: 
--- Начинаем сканирование свечи 60 ---


✅ Свеча 59 | O:0.5284 H:0.5286 L:0.5283 C:0.5285


DEBUG: ['C:\\Program Files\\Tesseract-OCR\\tesseract.exe', 'C:\\Users\\abuta\\Desktop\\mexc_bot\\temp_candle_60.png', 'C:\\Users\\abuta\\AppData\\Local\\Temp\\tess_c46d_6th', '-l', 'eng', '--psm', '6', '-c', 'tessedit_char_whitelist=0123456789.OHLCW', 'txt']
DEBUG: Tesseract: Сырой текст после Otsu-обработки: '0.5286000.5286LO.52862C0.52840.00010.045'
DEBUG: Скриншот сохранен: capture_candle_60_1762469393.png
DEBUG: Парсер: Очищенный текст (только числа): '0.5286000.5286  .52862 0.52840.00010.045'
DEBUG: Парсер: Найдено 6 чисел. Присваиваем по порядку O, H, L, C.
INFO: УСПЕХ: Все 4 значения распознаны.


✅ Свеча 60 | O:0.5286 H:0.5286 L:0.5286 C:0.5284

==================== СКАНИРОВАНИЕ ЗАВЕРШЕНО ====================


In [3]:
import pyautogui
import pytesseract
from PIL import Image
import time
import os

# --- НАСТРОЙКИ OCR ---
# Укажите полный путь к исполняемому файлу tesseract.exe
# Пример для Windows: r'C:\Program Files\Tesseract-OCR\tesseract.exe'
# Пример для macOS/Linux: r'/usr/local/bin/tesseract' (или где он установлен)
tesseract_cmd_path = r'C:\Program Files\Tesseract-OCR\tesseract.exe' 
pytesseract.pytesseract.tesseract_cmd = tesseract_cmd_path

# Координаты области для распознавания (left, top, width, height)
# Используем ваши новые координаты: [10, 580, 62, 20]
OCR_REGION = (10, 580, 62, 20)

# Язык для распознавания (например, 'eng' для английского, 'rus' для русского)
# Убедитесь, что соответствующий языковой пакет установлен в Tesseract.
# Если вы устанавливали Tesseract через установщик, английский язык обычно есть по умолчанию.
OCR_LANGUAGE = 'eng' 

# --- ФУНКЦИЯ ДЛЯ ТЕСТИРОВАНИЯ OCR ---
def test_ocr_region(region: tuple, lang: str = 'eng', confidence_threshold: int = 60):
    """
    Делает скриншот указанной области и пытается распознать текст.
    Выводит распознанный текст и уровень доверия.
    """
    print(f"--- Тестирование OCR в области {region} ---")
    try:
        # 1. Делаем скриншот указанной области
        screenshot_path = "temp_ocr_screenshot.png"
        pyautogui.screenshot(screenshot_path, region=region)
        print(f"Скриншот области сохранен в: {screenshot_path}")

        # 2. Загружаем скриншот и выполняем OCR
        img = Image.open(screenshot_path)
        
        # Конфигурация для Tesseract
        # '--oem 3 --psm 7' часто используется для распознавания одной строки текста
        # '--psm 7' - Treat the image as a single text line.
        # '--psm 6' - Assume a single uniform block of text.
        # '--oem 3' - Use LSTM and legacy engine (or just LSTM if available)
        config = f'--oem 3 --psm 7 -l {lang}' 
        
        # Используем image_to_data для получения текста и уровня доверия
        data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT, config=config)
        
        recognized_text = ""
        highest_conf = 0
        
        # Итерируем по словам, чтобы найти наиболее уверенное распознавание
        for i in range(len(data['text'])):
            word = data['text'][i].strip()
            conf = int(data['conf'][i])
            
            if word and conf > -1: # Пропускаем пустые строки и нераспознанные
                if conf > highest_conf:
                    highest_conf = conf
                recognized_text += (word + " ") # Собираем весь распознанный текст
        
        recognized_text = recognized_text.strip()
        
        print(f"\nРаспознанный текст: '{recognized_text}'")
        print(f"Максимальный уровень доверия: {highest_conf}%")

        if recognized_text and highest_conf >= confidence_threshold:
            print(f"✅ OCR успешно распознал текст с достаточным доверием.")
            return recognized_text
        elif recognized_text and highest_conf < confidence_threshold:
            print(f"⚠️ OCR распознал текст, но уровень доверия ({highest_conf}%) ниже порога ({confidence_threshold}%).")
        else:
            print("❌ OCR не распознал текст в указанной области.")
        
        return recognized_text

    except pytesseract.TesseractNotFoundError:
        print(f"ОШИБКА: Tesseract-OCR не найден по пути: {tesseract_cmd_path}")
        print("Пожалуйста, убедитесь, что Tesseract установлен и путь указан верно.")
    except FileNotFoundError:
        print(f"ОШИБКА: Не удалось сохранить скриншот или найти Tesseract. Проверьте права доступа и путь.")
    except Exception as e:
        print(f"Произошла непредвиденная ошибка: {e}")
    finally:
        # Удаляем временный файл скриншота
        if os.path.exists(screenshot_path):
            # os.remove(screenshot_path)
            print(f"Временный скриншот {screenshot_path} удален.")
    
    return ""

# --- ЗАПУСК ТЕСТА ---
if __name__ == "__main__":
    # Убедитесь, что окно с текстом, который вы хотите распознать, находится на переднем плане
    # и текст виден в области (10, 580, 62, 20).
    print("Подготовьте окно с текстом. Тест начнется через 3 секунды...")
    time.sleep(3) 
    
    recognized_value = test_ocr_region(OCR_REGION, lang=OCR_LANGUAGE)
    
    if "JN" in recognized_value: # Пример проверки, замените на нужный вам текст
        print("\nТЕСТ УСПЕШЕН: 'JN' найден в распознанном тексте!")
    else:
        print("\nТЕСТ НЕУДАЧЕН: 'JN' не найден или текст не распознан.")

Подготовьте окно с текстом. Тест начнется через 3 секунды...
--- Тестирование OCR в области (10, 580, 62, 20) ---
Скриншот области сохранен в: temp_ocr_screenshot.png

Распознанный текст: 'sitic'
Максимальный уровень доверия: 60%
✅ OCR успешно распознал текст с достаточным доверием.
Временный скриншот temp_ocr_screenshot.png удален.

ТЕСТ НЕУДАЧЕН: 'JN' не найден или текст не распознан.


In [3]:
import pyautogui
from PIL import Image
import pytesseract
import cv2 
import numpy as np 
import logging
import os
import time
import yaml # Библиотека для работы с YAML

# --- ⚙️ НАСТРОЙКА ЛОГИРОВАНИЯ ---
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# --- 🚩 ФУНКЦИЯ ЧТЕНИЯ КОНФИГА ---

def load_config(file_path='clicker_config.yaml'):
    """Читает и возвращает конфигурацию из YAML файла."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return yaml.safe_load(f)
    except FileNotFoundError:
        logger.critical(f"Файл конфигурации '{file_path}' не найден.")
        return None
    except Exception as e:
        logger.critical(f"Ошибка чтения YAML файла: {e}")
        return None

# --- ⚙️ ЗАГРУЗКА И ИНИЦИАЛИЗАЦИЯ ---

config = load_config()

if config:
    # Инициализируем Tesseract
    try:
        TESSERACT_PATH = config['OCR']['TESSERACT_PATH']
        pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH
    except KeyError:
        logger.critical("Не найден путь к Tesseract (OCR:TESSERACT_PATH). Проверьте config.yaml.")
        exit()

    # Параметры для распознавания Positions(N)
    POSITIONS_HEADER_REGION = tuple(config['OCR']['POSITIONS_HEADER_REGION'])
    POSITIONS_OCR_CONFIG = config['OCR']['POSITIONS_OCR_CONFIG']
    
    # Параметры предобработки
    SCALE_FACTOR = config['IMAGE_PROCESSING']['SCALE_FACTOR']
    BLUR_KERNEL_SIZE = tuple(config['IMAGE_PROCESSING']['BLUR_KERNEL_SIZE'])
    
    # Задержка перед сканированием
    SCAN_DELAY = 5 
    
    logger.info("Конфигурация успешно загружена. Tesseract инициализирован.")
else:
    logger.critical("Не удалось загрузить конфигурацию. Завершение работы.")
    exit()

# --- 🚩 ФУНКЦИЯ РАСПОЗНАВАНИЯ ---

def recognize_positions_count() -> str:
    """
    Распознает и возвращает сырой текст из области заголовка вкладки Positions,
    например: "Positions(0)".
    """
    logger.info(f"Добавляем задержку {SCAN_DELAY} сек. перед сканированием для стабилизации.")
    time.sleep(SCAN_DELAY) # Задержка 5 секунд
    
    logger.info("Начало распознавания заголовка Positions...")
    
    try:
        # 1. Захват скриншота
        screenshot_pil = pyautogui.screenshot(region=POSITIONS_HEADER_REGION)
        img_cv = np.array(screenshot_pil)
        img_cv = cv2.cvtColor(img_cv, cv2.COLOR_RGB2BGR) 
        
        # 2. Предобработка изображения (Upscale, Blur, Otsu)
        h, w, _ = img_cv.shape
        img_upscaled = cv2.resize(img_cv, (int(w * SCALE_FACTOR), int(h * SCALE_FACTOR)), interpolation=cv2.INTER_LINEAR)
        
        gray = cv2.cvtColor(img_upscaled, cv2.COLOR_BGR2GRAY)
        gray_blurred = cv2.GaussianBlur(gray, BLUR_KERNEL_SIZE, 0)
        _, binary = cv2.threshold(gray_blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        
        # Сохранение временного файла для OCR
        temp_img_path = f"temp_positions_header.png"
        cv2.imwrite(temp_img_path, binary)
        
        # 3. OCR
        # Используем специальный конфиг для заголовка позиций
        raw_text = pytesseract.image_to_string(temp_img_path, config=POSITIONS_OCR_CONFIG).strip()
        
        # Очистка временного файла
        os.remove(temp_img_path)
        
        logger.info(f"Распознанный сырой текст: '{raw_text}'")
        return raw_text
        
    except Exception as e:
        logger.error(f"Ошибка при распознавании заголовка Positions: {e}")
        # Сохранение скриншота для диагностики при ошибке
        screenshot_pil.save(f"debug_positions_error_{int(time.time())}.png")
        return "" # Возвращаем пустую строку при ошибке

# --- ПРИМЕР ИСПОЛЬЗОВАНИЯ В ЛОГИКЕ ОТКРЫТИЯ ПОЗИЦИИ ---

def check_and_open_position():
    """Проверочная логика: открывать позицию, только если Positions(0)."""
    
    header_text = recognize_positions_count().lower().replace(" ", "") # Нормализация
    
    # 3. Улучшенная логика проверки
    if not header_text:
        logger.warning("⚠️ Надпись с количеством позиций не распознана. Проверьте координаты или настройки OCR.")
        
    elif header_text == "positions(0)":
        logger.info("✅ Все позиции закрыты (Positions(0)). Продолжаем открытие новой позиции.")
        # --- ЗДЕСЬ БУДЕТ ЛОГИКА КЛИКЕРА ДЛЯ ОТКРЫТИЯ ---
        pass
        
    else:
        logger.warning(f"❌ Обнаружены открытые позиции. Заголовок: '{header_text}'. Ждем или обрабатываем ошибку.")
        
        
if __name__ == "__main__":
    # Тест-запуск функции
    check_and_open_position()

INFO: Конфигурация успешно загружена. Tesseract инициализирован.
INFO: Добавляем задержку 5 сек. перед сканированием для стабилизации.
INFO: Начало распознавания заголовка Positions...
INFO: Распознанный сырой текст: 'Positions(0)'
INFO: ✅ Все позиции закрыты (Positions(0)). Продолжаем открытие новой позиции.


In [10]:
import pyautogui
from PIL import Image
import pytesseract
import cv2 
import numpy as np 
import logging
import os
import time
import yaml 

# --- ⚙️ НАСТРОЙКА ЛОГИРОВАНИЯ ---
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# --- 🚩 ФУНКЦИЯ ЧТЕНИЯ КОНФИГА (Упрощенная для примера) ---

def load_config(file_path='clicker_config.yaml'):
    """Читает и возвращает конфигурацию из YAML файла."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return yaml.safe_load(f)
    except Exception as e:
        logger.critical(f"Ошибка чтения YAML файла или файл не найден: {e}")
        return None

# --- ⚙️ ЗАГРУЗКА И ИНИЦИАЛИЗАЦИЯ ---

config = load_config()

# Задержка сканирования кнопки
CONFIRM_SCAN_DELAY = 5 

if config:
    # Инициализация Tesseract
    try:
        pytesseract.pytesseract.tesseract_cmd = config['OCR']['TESSERACT_PATH']
    except KeyError:
        logger.critical("Не найден путь к Tesseract. Проверьте config.yaml.")
        exit()

    # Параметры для распознавания кнопки "Confirm"
    CONFIRM_BUTTON_REGION = tuple(config['POPUP_CONFIRMATION']['CONFIRM_BUTTON_REGION'])
    CONFIRM_OCR_CONFIG = config['POPUP_CONFIRMATION']['CONFIRM_OCR_CONFIG']
    BUTTON_CONFIRM_COORDS = config['POPUP_CONFIRMATION']['BUTTON_CONFIRM_COORDS']
    
    # Параметры предобработки
    SCALE_FACTOR = config['IMAGE_PROCESSING']['SCALE_FACTOR']
    BLUR_KERNEL_SIZE = tuple(config['IMAGE_PROCESSING']['BLUR_KERNEL_SIZE'])
    
else:
    logger.critical("Не удалось загрузить конфигурацию. Завершение работы.")
    exit()

# --- 🚩 ФУНКЦИЯ РАСПОЗНАВАНИЯ КНОПКИ ---

def recognize_confirm_button() -> str:
    """
    Распознает текст 'Confirm' в области всплывающего окна.
    Возвращает 'Confirm' при успехе, иначе пустую строку.
    """
    logger.info(f"Добавляем задержку {CONFIRM_SCAN_DELAY} сек. перед сканированием для стабилизации.")
    time.sleep(CONFIRM_SCAN_DELAY)
    
    logger.info("Начало распознавания кнопки 'Confirm'...")
    
    try:
        screenshot_pil = pyautogui.screenshot(region=CONFIRM_BUTTON_REGION)
        img_cv = np.array(screenshot_pil)
        img_cv = cv2.cvtColor(img_cv, cv2.COLOR_RGB2BGR) 
        
        # 1. Предобработка изображения (Upscale, Blur)
        h, w, _ = img_cv.shape
        img_upscaled = cv2.resize(img_cv, (int(w * SCALE_FACTOR), int(h * SCALE_FACTOR)), interpolation=cv2.INTER_LINEAR)
        gray = cv2.cvtColor(img_upscaled, cv2.COLOR_BGR2GRAY)
        gray_blurred = cv2.GaussianBlur(gray, BLUR_KERNEL_SIZE, 0)
        
        # 2. УЛУЧШЕННАЯ Бинаризация (Инверсия + Простой Порог)
        # 2.1 Инвертируем изображение (белый текст -> черный текст)
        inverted = cv2.bitwise_not(gray_blurred)
        
        # 2.2 Применяем простой порог для отделения черного текста от светлого фона
        _, binary = cv2.threshold(inverted, 100, 255, cv2.THRESH_BINARY) 
        
        temp_img_path = f"temp_confirm_button.png"
        cv2.imwrite(temp_img_path, binary)
        
        # 3. OCR
        raw_text = pytesseract.image_to_string(temp_img_path, config=CONFIRM_OCR_CONFIG).strip()
        os.remove(temp_img_path)
        
        # Всегда сохраняем скриншот после OCR
        screenshot_filename = f"capture_confirm_button_{int(time.time())}.png"
        screenshot_pil.save(screenshot_filename)
        logger.debug(f"Скриншот сохранен: {screenshot_filename}")
        
        logger.info(f"Распознанный сырой текст: '{raw_text}'")
        
        # 4. Проверка результата (гибкое совпадение)
        normalized_text = raw_text.lower().replace(" ", "")
        
        if "confirm" in normalized_text or "confir" in normalized_text:
            logger.info("✅ Кнопка 'Confirm' успешно распознана.")
            return "Confirm"
        else:
            return ""
        
    except Exception as e:
        logger.error(f"Ошибка при распознавании кнопки 'Confirm': {e}")
        if 'screenshot_pil' in locals():
             screenshot_pil.save(f"debug_confirm_error_{int(time.time())}.png")
        return ""

# --- ПРИМЕР ИСПОЛЬЗОВАНИЯ В ЛОГИКЕ ---

def handle_stop_order_confirmation():
    """Проверяет наличие кнопки Confirm и нажимает ее."""
    
    result = recognize_confirm_button()
    
    if result == "Confirm":
        coords = BUTTON_CONFIRM_COORDS
        logger.warning(f"❗ Обнаружено предупреждение о цене. Нажимаем 'Confirm' по координатам: {coords}")
        # pyautogui.click(coords[0], coords[1])
        time.sleep(1) # Небольшая задержка после клика
        return True
    else:
        logger.info("Кнопка 'Confirm' не обнаружена. Продолжаем.")
        return False

if __name__ == "__main__":
    handle_stop_order_confirmation()

INFO: Добавляем задержку 5 сек. перед сканированием для стабилизации.
INFO: Начало распознавания кнопки 'Confirm'...
INFO: Распознанный сырой текст: 'Confirm'
INFO: ✅ Кнопка 'Confirm' успешно распознана.
